In [14]:
# -*- coding: utf-8 -*-
from pathlib import Path
import os
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import gcamreader

In [15]:
def convert_to_mt(row):
    val, unit = row['value'], row['Units']
    if unit == 'Tg':
        return val
    elif unit == 'Gg':
        return val * 1e-3
    elif unit == 'MTC':
        return val * (44.009 / 12.011)
    else:
        raise ValueError(f"Unknown unit: {unit}")

# AR5 100‑yr GWP (no climate–carbon feedbacks)
GWP_AR5 = {
    'CO2':      1,      
    'CH4':     28,      
    'CH4_AGR': 28,
    'CH4_AWB': 28,
    'N2O':    265,      
    'N2O_AGR':265,
    'N2O_AWB':265,
    'HFC125': 3170,     
    'HFC134a':1300,     
    'HFC143a':4800,     
    'HFC23': 12400,     
    'HFC32':   677,     
    'HFC43':  1650,     
    'HFC227ea':3350,    
    'HFC236fa':8060,    
    'SF6':   23500,     
    'C2F6':  11100,     
    'CF4':    6630,     
}

In [16]:
# =========================================================
# Config
# =========================================================
PROJECT_PATH   = Path("/data/project/tae/gcam-core")
DB_REL_PATH    = "../output"
DB_FILE        = "database_basexdb_korea_2035_v4"
QUERY_FILE     = Path("..") / "output" / "queries" / "Main_queries.xml"

REGION = "South Korea"
SCENARIOS = [
    "Current-Policies-Med", "High-Ambition-Med",
    "Current-Policies-Low", "High-Ambition-Low",
    "Current-Policies-High","High-Ambition-High"
]

# Query indices
Q_CO2   = 262
Q_NONCO2= 270

# Historical emissions (MtCO2e)
EMISS_2018 = 742.3

# Paths
EXTDATA_XLSX = "./extdata/gir2025.xlsx"

# Output figs
FIG_TOTAL = "./fig/total_emission.png"
FIG_HIST  = "./fig/korea_hist.png"

In [17]:
# =========================================================
# DB helpers
# =========================================================
def connect_db():
    return gcamreader.LocalDBConn(DB_REL_PATH, DB_FILE)

def run_query(conn, q_idx):
    queries = gcamreader.parse_batch_query(os.fspath(QUERY_FILE))
    q = queries[q_idx]
    df = conn.runQuery(q, scenarios=SCENARIOS, regions=[REGION])
    df["scenario"] = df["scenario"].str.split(",").str[0]
    return df

In [18]:
# =========================================================
# Historical loaders
# =========================================================
def read_gir_sheet(sheet):
    return pd.read_excel(
        EXTDATA_XLSX, sheet_name=sheet, skiprows=3,
        index_col=1, skipfooter=9, engine="openpyxl"
    ).transpose().iloc[1:, ]

def load_historical():
    # gas-specific totals
    dfCO2 = read_gir_sheet("CO2")
    dfCH4 = read_gir_sheet("CH4")
    dfN2O = read_gir_sheet("N2O")
    dfSF6 = read_gir_sheet("SF6")
    dfHFC = read_gir_sheet("HFCs")
    dfPFC = read_gir_sheet("PFCs")

    gas = pd.DataFrame({
        "CO2": dfCO2["총배출량"],
        "CH4": dfCH4["총배출량"],
        "N2O": dfN2O["총배출량"],
        "F-Gases": dfSF6["총배출량"] + dfHFC["총배출량"] + dfPFC["총배출량"],
    }).reset_index(names=["Year"])

    # stacked (Gg → Mt)
    gas["CO2_stack"] = gas["CO2"] / 1000
    gas["CH4_stack"] = (gas["CO2"] + gas["CH4"]) / 1000
    gas["N2O_stack"] = (gas["CO2"] + gas["CH4"] + gas["N2O"]) / 1000
    gas["FGas_stack"] = (gas["CO2"] + gas["CH4"] + gas["N2O"] + gas["F-Gases"]) / 1000

    # “Net emissions” timeseries for the scenario legend line
    net = pd.read_excel(
        EXTDATA_XLSX, skiprows=3, index_col=1, skipfooter=9, engine="openpyxl"
    ).transpose().iloc[1:, ].reset_index()
    # net has columns: 'index' (year), '순배출량' (Gg CO2e)
    net.rename(columns={"index": "Year"}, inplace=True)
    net["NetMt"] = net["순배출량"] / 1000
    return gas, net

In [19]:
# =========================================================
# Model outputs → MtCO2e (AR5)
# =========================================================
def build_model_mtco2e(conn):
    df_co2   = run_query(conn, Q_CO2)
    df_co2["GHG"] = "CO2"

    df_nonco2= run_query(conn, Q_NONCO2)

    df = pd.concat([df_co2, df_nonco2], ignore_index=True)
    df["emiss(MT)"] = df.apply(convert_to_mt, axis=1)  # uses your utils.convert_to_mt
    df["gwpAr5"]    = df["GHG"].map(GWP_AR5).astype(float)
    df["MTCO2eq"]   = df["emiss(MT)"] * df["gwpAr5"]
    return df

In [20]:
(41.0 + 47.8) / 2

44.4

In [21]:
# =========================================================
# LULUCF negative-emissions assumptions (MtCO2e, subtracted)
# =========================================================
LULUCF_NEG_EMISSIONS = {
    "Current-Policies-Med":  {2005: 57.5, 2010: 57.3, 2015: 47.8, 2020: 38.8, 2025: 41.0, 2030: 41.0, 2035: 41.0},
    "High-Ambition-Med": {2005: 57.5, 2010: 57.3, 2015: 47.8, 2020: 38.8, 2025: 41.0, 2030: 38.8, 2035: 47.8},
    "Current-Policies-High": {2005: 57.5, 2010: 57.3, 2015: 47.8, 2020: 38.8, 2025: 41.0, 2030: 15.2, 2035: 0.0},
    "High-Ambition-High":{2005: 57.5, 2010: 57.3, 2015: 47.8, 2020: 38.8, 2025: 41.0, 2030: 41.0, 2035: 41.0},
    "Current-Policies-Low":  {2005: 57.5, 2010: 57.3, 2015: 47.8, 2020: 38.8, 2025: 41.0, 2030: 28.4, 2035: 26.4},
    "High-Ambition-Low": {2005: 57.5, 2010: 57.3, 2015: 47.8, 2020: 38.8, 2025: 41.0, 2030: 44.4, 2035: 47.8},
}

def apply_lulucf_sinks(df_out):
    """
    Apply LULUCF negative-emissions (sinks) to totals.

    Parameters
    ----------
    df_out : DataFrame indexed by ['scenario','Year'] with a 'value' column in MtCO2e
        Represents gross economy-wide emissions (excl. intl. aviation/shipping).

    Notes
    -----
    Values in LULUCF_NEG_EMISSIONS are *MtCO2e sinks* (positive numbers),
    which are subtracted from gross emissions: net = gross - sink.
    """
    for scen, year_map in LULUCF_NEG_EMISSIONS.items():
        for yr, sink_mtco2e in year_map.items():
            key = (scen, yr)
            if key in df_out.index:
                df_out.loc[key, "value"] -= sink_mtco2e

In [22]:
# =========================================================
# Plot builders
# =========================================================
def add_gas_stack(fig, gas_df):
    fig.add_trace(go.Scatter(x=[None], y=[None], mode='lines',
                             line=dict(color='rgba(0,0,0,0)'),
                             name='<br><br><br><b>Gas</b>',
                             showlegend=True, hoverinfo='skip'))
    fig.add_trace(go.Scatter(x=gas_df["Year"], y=gas_df["CO2_stack"],
                             mode='lines', name='CO2', fill='tozeroy',
                             line=dict(width=0.5, color='grey')))
    fig.add_trace(go.Scatter(x=gas_df["Year"], y=gas_df["CH4_stack"],
                             mode='lines', name='CH4', fill='tonexty',
                             line=dict(width=0.5, color='green')))
    fig.add_trace(go.Scatter(x=gas_df["Year"], y=gas_df["N2O_stack"],
                             mode='lines', name='N2O', fill='tonexty',
                             line=dict(width=0.5, color='purple')))
    fig.add_trace(go.Scatter(x=gas_df["Year"], y=gas_df["FGas_stack"],
                             mode='lines', name='F-Gases', fill='tonexty',
                             line=dict(width=0.5, color='yellow')))

def add_scenario_lines(fig, df_out, cp_name, ep_name, start_year=2025):
    # separator in legend
    fig.add_trace(go.Scatter(x=[None], y=[None], mode='lines',
                             line=dict(color='rgba(0,0,0,0)'),
                             name='<br><br><br><b>Scenario</b>',
                             showlegend=True, hoverinfo='skip'))
    # historical net
    # (added outside with add_hist_net for clarity)

    # scenario lines
    mask_cp = (df_out["scenario"] == cp_name) & (df_out["Year"] >= start_year)
    mask_ep = (df_out["scenario"] == ep_name) & (df_out["Year"] >= start_year)
    fig.add_trace(go.Scatter(
        x=df_out.loc[mask_cp, "Year"], y=df_out.loc[mask_cp, "value"],
        mode='lines', name='Current Policies',
        line=dict(color='#636EFA', width=1.5)
    ))
    fig.add_trace(go.Scatter(
        x=df_out.loc[mask_ep, "Year"], y=df_out.loc[mask_ep, "value"],
        mode='lines', name='High Ambition',
        line=dict(color='#00CC96', width=1.5)
    ))

def add_scenario_bands(fig, df_out, low_scen, high_scen, fillcolor):
    yrs  = df_out.loc[df_out["scenario"] == low_scen, "Year"].values
    low  = df_out.loc[df_out["scenario"] == low_scen, "value"].values
    high = df_out.loc[df_out["scenario"] == high_scen, "value"].values
    if len(yrs) and len(low) and len(high) and len(low) == len(high):
        fig.add_trace(go.Scatter(
            x=list(yrs) + list(yrs[::-1]),
            y=list(high) + list(low[::-1]),
            fill='toself', fillcolor=fillcolor,
            line=dict(color='rgba(255,255,255,0)'),
            hoverinfo="skip", showlegend=False
        ))

def style_axes(fig, x_range, y_range, x_ticks, y_ticks, x_title="Year", y_title="Emission (MtCO2e)"):
    fig.update_layout(
        legend_traceorder="normal", legend_title='', legend_font_size=15,
        plot_bgcolor='rgba(0,0,0,0)', width=1200, height=700,
        xaxis=dict(showgrid=False, title=x_title, title_font_size=25,
                   tickvals=x_ticks, tickfont_size=15, range=x_range),
        yaxis=dict(showgrid=False, title=y_title, title_font_size=25,
                   tickvals=y_ticks, tickfont_size=15, range=y_range),
    )

def add_grid(fig, x0, x1, y0, y1, x_step=5, y_step=100):
    for x in range(int(x0), int(x1)+1, x_step):
        fig.add_shape(type="line", x0=x, x1=x, y0=y0, y1=y1,
                      line=dict(color="LightGrey", width=1, dash="dash"), layer='below')
    for y in range(int(y0), int(y1)+1, y_step):
        fig.add_shape(type="line", x0=x0, x1=x1, y0=y, y1=y,
                      line=dict(color="LightGrey", width=1, dash="dash"), layer='below')
    fig.add_shape(type="line", x0=x0, x1=x1, y0=0, y1=0,
                  line=dict(color="LightGrey", width=1, dash="dash"), layer='above')

In [23]:
conn = connect_db()

Database scenarios: Current-Policies-Med, High-Ambition-Med, Current-Policies-High, High-Ambition-High, Current-Policies-Low, High-Ambition-Low, High-Ambition-Med, High-Ambition-Med, High-Ambition-Med, High-Ambition-Med, Current-Policies-Med, High-Ambition-Low, Current-Policies-Low, High-Ambition-High, Current-Policies-High


In [24]:
# ----- Load model outputs → MtCO2e -----
dfGHG = build_model_mtco2e(conn)
# drop international shipping/aviation
dfOut = (
    dfGHG[~dfGHG["sector"].isin(["trn_aviation_intl", "trn_shipping_intl"])]
    .groupby(["scenario", "Year"], as_index=False)["MTCO2eq"].sum()
    .rename(columns={"MTCO2eq": "value"})
)

# LULUCF sinks
dfOut = dfOut.set_index(["scenario", "Year"])
apply_lulucf_sinks(dfOut)
dfOut = dfOut.reset_index()

# Precompute 2035 reductions vs 2018
def pct_red(v): return (1 - (v / EMISS_2018)) * 100
cp_med_2035 = dfOut.set_index(["scenario", "Year"])["value"].get(("Current-Policies-Med", 2035))
ep_med_2035 = dfOut.set_index(["scenario", "Year"])["value"].get(("High-Ambition-Med", 2035))
red_cp = pct_red(cp_med_2035) if cp_med_2035 is not None else np.nan
red_ep = pct_red(ep_med_2035) if ep_med_2035 is not None else np.nan

# ----- Historical data -----
dfHistGas, dfHistNet = load_historical()

In [25]:
dfOut

,scenario,Year,value
0,Current-Policies-High,1975,54.590644
1,Current-Policies-High,1990,295.171616
2,Current-Policies-High,2005,531.334536
3,Current-Policies-High,2010,637.680605
4,Current-Policies-High,2015,702.955896
5,Current-Policies-High,2020,670.926445
6,Current-Policies-High,2025,646.647275
7,Current-Policies-High,2030,625.644242
8,Current-Policies-High,2035,571.602432
9,Current-Policies-Low,1975,54.590644


In [26]:
# ============================
# Figure 1: Total emissions with bands & annotations
# ============================
fig = go.Figure()
add_gas_stack(fig, dfHistGas)

# NDC marker (60% of 2018 at 2030)
fig.add_trace(go.Scatter(
    x=[2030], y=[783.8 * 0.6],
    mode='markers+text',
    marker=dict(color='#750D86', size=7, symbol='triangle-up'),
    text='2030 NDC', textposition="middle left",
    textfont=dict(size=15, color="#750D86"),
    showlegend=False
))
# 2018 reference
fig.add_trace(go.Scatter(
    x=[2018], y=[EMISS_2018],
    mode='markers+text',
    marker=dict(color='black', size=7),
    text=f'2018: {EMISS_2018} MtCO2e',
    textposition='middle right',
    textfont=dict(size=15),
    showlegend=False
))
# Historical net line
fig.add_trace(go.Scatter(
    x=dfHistNet["Year"], y=dfHistNet["NetMt"],
    mode='lines', name='Historical Data',
    line=dict(color='black', width=1)
))

fig.add_trace(go.Scatter(
    x=[2023, 2024],
    y=[666.3 + 1.5, 651.4 + 1.5],
    mode='markers',                          # only points
    # name='Net (Provisional)',            # shows in legend
    marker=dict(color='black', size=7, symbol='x'),
    showlegend=False
))

# Scenario lines + bands
add_scenario_lines(fig, dfOut, "Current-Policies-Med", "High-Ambition-Med", start_year=2025)
add_scenario_bands(fig, dfOut, "Current-Policies-Low", "Current-Policies-High", fillcolor='rgba(99,110,250,0.2)')
add_scenario_bands(fig, dfOut, "High-Ambition-Low", "High-Ambition-High", fillcolor='rgba(0,204,150,0.2)')

# Layout & grid
style_axes(fig, x_range=[1987, 2045], y_range=[-30, 830],
            x_ticks=list(range(1990, 2040, 5)),
            y_ticks=list(range(0, 801, 100)))
# highlight 2025–2035
fig.add_vrect(x0=2025, x1=2035, fillcolor="LightBlue", opacity=0.1, layer="below", line_width=0)
# border
fig.add_shape(type="rect", xref="paper", yref="paper",
                x0=0, x1=1, y0=0, y1=1, line=dict(color="black", width=1), layer="above")
# grid
add_grid(fig, 1985, 2040, -100, 900)

# Reduction annotations at 2035
x_center = 2040
fig.add_annotation(x=x_center - 0.15, y=cp_med_2035,
                    text=f"<b>Current<br>Policies<br>-{red_cp:.1f}%</b>",
                    showarrow=False, font=dict(size=14, color="#636EFA"))
fig.add_annotation(x=x_center + 0.15, y=ep_med_2035,
                    text=f"<b>High<br>Ambition<br>-{red_ep:.1f}%</b>",
                    showarrow=False, font=dict(size=14, color="#00CC96"))

# pio.write_image(fig, FIG_TOTAL, width=1200, height=700, scale=2)
fig

In [27]:
# ============================
# Figure 2: Historical + slopes (1990→2018; 2018→2050)
# ============================
fig2 = go.Figure()
add_gas_stack(fig2, dfHistGas)

# NDC marker & 2018 point
fig2.add_trace(go.Scatter(
    x=[2030], y=[EMISS_2018 * 0.6],
    mode='markers+text',
    marker=dict(color='#750D86', size=7, symbol='triangle-up'),
    text='2030 NDC', textposition="middle left",
    textfont=dict(size=15, color="#750D86"),
    showlegend=False
))
fig2.add_trace(go.Scatter(
    x=[2018], y=[EMISS_2018],
    mode='markers+text',
    marker=dict(color='black', size=7),
    text=f'2018: {EMISS_2018} MtCO2e',
    textposition='middle right',
    textfont=dict(size=15),
    showlegend=False
))
# Scenario separator + historical net
fig2.add_trace(go.Scatter(
    x=[None], y=[None], mode='lines',
    line=dict(color='rgba(0,0,0,0)'),
    name='<br><br><br><b>Scenario</b>', showlegend=True, hoverinfo='skip'
))
fig2.add_trace(go.Scatter(
    x=dfHistNet["Year"], y=dfHistNet["NetMt"],
    mode='lines', name='Historical Data',
    line=dict(color='black', width=1)
))

style_axes(fig2, x_range=[1987, 2055], y_range=[-30, 830],
            x_ticks=list(range(1990, 2055, 5)),
            y_ticks=list(range(0, 801, 100)))
fig2.add_vrect(x0=2025, x1=2055, fillcolor="LightBlue", opacity=0.1, layer="below", line_width=0)
fig2.add_shape(type="rect", xref="paper", yref="paper",
                x0=0, x1=1, y0=0, y1=1, line=dict(color="black", width=1), layer="above")
add_grid(fig2, 1987, 2055, -30, 830)

# Trend (1990→2018)
y1990 = 310.57
slope_hist = (EMISS_2018 - y1990) / (2018 - 1990)
fig2.add_shape(type="line", xref="x", yref="y",
                x0=1990, y0=y1990, x1=2018, y1=EMISS_2018,
                line=dict(color="black", dash="dash", width=2), layer="above")
fig2.add_annotation(x=2004, y=(y1990 + EMISS_2018)/2,
                    text=f"Slope ≈ {slope_hist:.2f} MtCO₂e/yr",
                    showarrow=True, arrowhead=2, ax=-80, ay=-40,
                    font=dict(size=14, color="black"))

# Net-zero line (2018→2050)
fig2.add_trace(go.Scatter(x=[2018, 2050], y=[EMISS_2018, 0],
                            mode="lines", line=dict(color="black", dash="dash", width=2),
                            name="2050 Net-Zero"))
slope_nz = (0 - EMISS_2018) / (2050 - 2018)
fig2.add_annotation(x=(2018 + 2050) / 2, y=EMISS_2018 / 2,
                    text=f"Slope ≈ {slope_nz:.2f} MtCO₂e/yr",
                    showarrow=True, arrowhead=2, ax=70, ay=-90,
                    font=dict(size=14, color="black"))

pio.write_image(fig2, FIG_HIST, width=1200, height=700, scale=2)

fig2

FileNotFoundError: [Errno 2] No such file or directory: 'fig/korea_hist.png'

In [ ]:
df_rates = (
    dfOut[dfOut['Year'] >= 2030]
    .pivot(index='scenario', columns='Year', values='value')
    .apply(lambda row: (1 - (row / 783.8)) * 100, axis=1)
)

print(df_rates.round(2))

Year                       2030  2035
scenario                             
Enhanced-Ambition-Med-61  40.57  59.9
